# Few-Shot Learning — Teaching by Example

## Introduction

**Few-shot prompting** is one of the most powerful techniques in prompt engineering. Instead of fine-tuning a model on thousands of labeled examples, you include a small number of examples directly in the prompt. The model recognizes the pattern from these examples and applies it to new inputs — a capability known as **in-context learning**.

This technique was first demonstrated at scale in the GPT-3 paper *"Language Models are Few-Shot Learners"* (Brown et al., 2020), and it remains a cornerstone of practical LLM usage today.

## What You'll Learn

- The difference between **zero-shot**, **one-shot**, and **few-shot** prompting
- How to construct few-shot prompts using the `FewShotSelector` and `Example` classes
- Strategies for **selecting** and **formatting** examples to maximize performance
- Best practices for choosing high-quality examples

## Prerequisites

- Basic Python knowledge (dataclasses, string formatting)
- Understanding of what an LLM prompt is (see Notebook 01)

## Learning Resources

- [Few-Shot Prompting Guide](https://www.promptingguide.ai/techniques/fewshot)
- [GPT-3 Paper — "Language Models are Few-Shot Learners"](https://arxiv.org/abs/2005.14165)
- [VIDEO: "Few-Shot Learning Explained"](https://www.youtube.com/watch?v=hE7eGew4eeg)

In [ ]:
# Import the few-shot learning components from AgentExplorr
from agentexplorr.prompt_engineering.few_shot import Example, FewShotSelector

# Verify imports
print("Example:", Example.__doc__.strip().split("\n")[0])
print("FewShotSelector:", FewShotSelector.__doc__.strip().split("\n")[0])
print("\nImports loaded successfully!")

## Zero-Shot vs One-Shot vs Few-Shot

The number of examples you include in a prompt determines the "shot" level:

| Strategy | # Examples | Description | When to Use |
|----------|-----------|-------------|-------------|
| **Zero-shot** | 0 | No examples — just an instruction. | Simple, well-defined tasks the model already understands (e.g., translation, summarization). |
| **One-shot** | 1 | A single example to demonstrate the format. | When the task format is non-obvious but the model mostly knows what to do. |
| **Few-shot** | 2-10 | Multiple examples showing the pattern. | When the task is novel, nuanced, or requires a specific output style. |

### Key Insight

> The **quality** of examples matters more than the **quantity**. Three well-chosen, diverse examples typically outperform ten repetitive ones. Choose examples that cover edge cases, represent different categories, and are similar to the inputs you expect in production.

In [ ]:
# ------------------------------------------------------------------
# Build a few-shot prompt for sentiment classification
# ------------------------------------------------------------------

# Step 1: Create a FewShotSelector and add labeled examples
selector = FewShotSelector()

sentiment_examples = [
    Example(
        input_text="I absolutely love this product! Best purchase I've ever made.",
        output_text="Positive",
        metadata={"category": "sentiment", "tone": "enthusiastic"},
    ),
    Example(
        input_text="The delivery was late and the item was broken. Very disappointed.",
        output_text="Negative",
        metadata={"category": "sentiment", "tone": "frustrated"},
    ),
    Example(
        input_text="It works fine. Nothing special, but gets the job done.",
        output_text="Neutral",
        metadata={"category": "sentiment", "tone": "indifferent"},
    ),
]

selector.add_examples(sentiment_examples)
print(f"Added {selector.size} examples to the selector.\n")

# Step 2: Build a complete few-shot prompt for a new input
prompt = selector.build_prompt(
    instruction="Classify the sentiment of the following text as Positive, Negative, or Neutral.",
    input_text="The food was okay but the service was really slow.",
    examples=sentiment_examples,  # use all 3 explicitly
    input_label="Text",
    output_label="Sentiment",
)

print("=" * 60)
print("GENERATED FEW-SHOT PROMPT")
print("=" * 60)
print(prompt)

## Example Selection Strategies

Not all examples are equally useful. The `FewShotSelector` supports several strategies for choosing which examples to include:

### 1. Random Selection
Pick N random examples from the pool. This is a surprisingly strong baseline because it naturally provides diversity. Use `selector.select_random(n=3)`.

### 2. Category-Based (Metadata Filtering)
Select examples matching a specific metadata key-value pair. This is useful when you want to ensure representation from certain categories — for instance, always including at least one "Positive" and one "Negative" example in a sentiment task.

### 3. Semantic Similarity (Advanced)
Pick examples that are most similar to the current input using embedding-based retrieval. This is the most effective strategy but requires a vector store (see the RAG module). The idea is that examples closer to the input give the model more relevant patterns to follow.

### Best Practices
- **Diversity**: Include examples from different categories or edge cases
- **Relevance**: Choose examples similar to the expected production inputs
- **Consistency**: Use the same format for all examples — the model learns the pattern from the structure

In [ ]:
# ------------------------------------------------------------------
# Formatting examples with different templates and labels
# ------------------------------------------------------------------

# Add more examples with metadata for filtering demos
selector_v2 = FewShotSelector()

topic_examples = [
    Example("Python is a great language for data science.", "Technology", metadata={"domain": "tech"}),
    Example("The stock market reached record highs today.", "Finance", metadata={"domain": "finance"}),
    Example("New study reveals benefits of Mediterranean diet.", "Health", metadata={"domain": "health"}),
    Example("SpaceX launched another batch of Starlink satellites.", "Technology", metadata={"domain": "tech"}),
    Example("Central bank raises interest rates by 0.25%.", "Finance", metadata={"domain": "finance"}),
]

selector_v2.add_examples(topic_examples)

# --- Demo 1: Random selection with custom labels ---
print("--- Random Selection (n=2, seed=42) ---")
random_examples = selector_v2.select_random(n=2, seed=42)
formatted = selector_v2.format_examples(
    examples=random_examples,
    input_label="Article",
    output_label="Topic",
)
print(formatted)

# --- Demo 2: Metadata-based filtering ---
print("\n--- Metadata Filter: domain='tech' ---")
tech_examples = selector_v2.select_by_metadata("domain", "tech")
formatted_tech = selector_v2.format_examples(
    examples=tech_examples,
    input_label="Headline",
    output_label="Category",
)
print(formatted_tech)

# --- Demo 3: Full prompt with filtered examples ---
print("\n--- Full Prompt Using Finance Examples ---")
finance_examples = selector_v2.select_by_metadata("domain", "finance")
full_prompt = selector_v2.build_prompt(
    instruction="Classify the following news headline into a topic category.",
    input_text="Apple announces record quarterly revenue driven by iPhone sales.",
    examples=finance_examples,
    input_label="Headline",
    output_label="Category",
)
print(full_prompt)

## Key Takeaways

1. **Few-shot prompting** lets you teach a model new tasks by including examples directly in the prompt — no fine-tuning required.
2. The `Example` dataclass pairs an input with its expected output, plus optional metadata for filtering and organization.
3. `FewShotSelector` manages a pool of examples and supports **random**, **metadata-based**, and (conceptually) **similarity-based** selection.
4. Custom **labels** (`input_label`, `output_label`) let you adapt the prompt format to any task — sentiment analysis, topic classification, entity extraction, and more.
5. **Quality over quantity**: 3-5 well-chosen, diverse examples usually outperform a larger set of repetitive ones.

## Next Steps

- **Notebook 03 — Structured Output**: Learn how to get LLMs to return valid JSON that you can parse into Python objects.
- **Experiment**: Try changing the examples above and observe how the generated prompt changes. What happens with only 1 example vs 5?
- **Advanced**: Combine few-shot prompting with RAG (Retrieval-Augmented Generation) to dynamically select the most relevant examples based on the user's input.